In [ ]:
from pathlib import Path
from zipfile import ZipFile
import html

import folium
import geopandas as gpd
import pandas as pd
import requests
from branca.colormap import linear


# ============================================================
# 1. MAP CONFIGURATION
# ============================================================

# Add or remove indicator keys here to control the map layers.
ACTIVE_INDICATORS = [
    "income",
    "council_renting",
    "owner_occupation",
    "house_price",
    "recorded_crime",
]

OUTPUT_FILE = "msoa_map.html"
MAP_KEY = "MSOA_CODE"

# All colour scales, ranges, labels and tooltip settings live here.
# Change these values without changing the map-building code below.
INDICATORS = {
    "income": {
        "label": "Total annual income",
        "column": "income_value",
        "palette": "YlGnBu_09",
        "minimum": 30_000,
        "maximum": 120_000,
        "legend_title": "Total annual income (£)",
        "format": "currency",
        "tooltip_alias": "Income (£):",
        "geo_source": "msoa",
    },
    "council_renting": {
        "label": "Council renting",
        "column": "council_rented_percent",
        "palette": "YlOrRd_06",
        "minimum": 0,
        "maximum": 50,
        "legend_title": "Council-rented households (%)",
        "format": "percent",
        "tooltip_alias": "Council-rented households (%):",
        "geo_source": "msoa",
    },
    "owner_occupation": {
        "label": "Owner occupation",
        "column": "owner_occupied_percent",
        "palette": "YlGnBu_06",
        "minimum": 20,
        "maximum": 80,
        "legend_title": "Owner-occupied households (%)",
        "format": "percent",
        "tooltip_alias": "Owner-occupied households (%):",
        "geo_source": "msoa",
    },
    "house_price": {
        "label": "Median house price",
        "column": "median_house_price",
        "palette": "YlGnBu_06",
        "minimum": 300_000,
        "maximum": 1_500_000,
        "legend_title": "Median price paid (£), year ending September 2025",
        "format": "currency",
        "tooltip_alias": "Median house price (£):",
        "geo_source": "msoa",
    },
    "recorded_crime": {
        "label": "Recorded crime (excluding fraud)",
        "column": "recorded_crime_rate_per_1000",
        "palette": "YlOrRd_06",
        "minimum": 50,
        "maximum": 100,
        "legend_title": "Police-recorded crime excluding fraud per 1,000 residents, year ending September 2025",
        "format": "number",
        "tooltip_alias": "Crime per 1,000 residents:",
        "geo_source": "pfa",
        "tooltip_fields": ["PFA_NAME", "recorded_crime_rate_per_1000"],
        "tooltip_aliases": ["Police force area:", "Crime per 1,000 residents:"],
    },
}


# ============================================================
# 2. GENERAL HELPERS
# ============================================================


def download_file(url, destination):
    """Download a file only when it is not already present."""
    if destination.exists():
        return

    response = requests.get(url, timeout=180)
    response.raise_for_status()
    destination.write_bytes(response.content)


def clean_code(series):
    """Standardise geography codes before joining datasets."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def to_number(series):
    """Convert numbers that may contain commas into numeric values."""
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce",
    )


def assert_unique(df, dataset_name):
    """Ensure each dataset has one row per MSOA."""
    if df[MAP_KEY].duplicated().any():
        raise ValueError(
            f"Duplicate {MAP_KEY} values found in {dataset_name}."
        )


# ============================================================
# 3. DATASET: MSOA BOUNDARIES
# ============================================================

BOUNDARY_URL = (
    "https://open-geography-portalx-ons.hub.arcgis.com/"
    "api/download/v1/items/"
    "61ff711e89ba4c24ae5dc8a487e422a8/"
    "geojson?layers=0"
)

BOUNDARY_FILE = Path("msoa_boundaries.geojson")


def load_boundary_dataset():
    """Download and load the official MSOA boundary polygons."""
    download_file(BOUNDARY_URL, BOUNDARY_FILE)

    boundaries = gpd.read_file(BOUNDARY_FILE)

    code_candidates = [
        column
        for column in boundaries.columns
        if "MSOA" in column.upper()
        and "CD" in column.upper()
    ]

    if not code_candidates:
        raise ValueError(
            "Could not find an MSOA code column in the boundaries."
        )

    boundaries = boundaries.rename(
        columns={code_candidates[0]: MAP_KEY}
    )

    boundaries[MAP_KEY] = clean_code(boundaries[MAP_KEY])
    assert_unique(boundaries, "MSOA boundaries")

    return boundaries


# ============================================================
# 4. DATASET: INCOME
# ============================================================

INCOME_URL = (
    "https://www.ons.gov.uk/file?uri="
    "/employmentandlabourmarket/peopleinwork/earningsandworkinghours/"
    "datasets/smallareaincomeestimatesformiddlelayersuperoutputareasenglandandwales/"
    "financialyearending2023/datasetfinal.xlsx"
)

INCOME_FILE = Path("income_fye2023.xlsx")


def load_income_dataset():
    """Load and standardise the ONS MSOA income data."""
    download_file(INCOME_URL, INCOME_FILE)

    income = pd.read_excel(
        INCOME_FILE,
        sheet_name="Total annual income",
        header=3,
    )

    income = income[
        [
            "MSOA code",
            "Local authority name",
            "Region name",
            "Total annual income (£)",
        ]
    ].copy()

    income = income.rename(
        columns={
            "MSOA code": MAP_KEY,
            "Total annual income (£)": "income_value",
        }
    )

    income[MAP_KEY] = clean_code(income[MAP_KEY])
    income["income_value"] = to_number(
        income["income_value"]
    )

    assert_unique(income, "income data")

    return income


# ============================================================
# 5. DATASET: CENSUS TENURE
# ============================================================

TS054_URL = (
    "https://www.nomisweb.co.uk/output/census/2021/"
    "census2021-ts054.zip"
)

TS054_FILE = Path("census2021-ts054.zip")
TS054_FOLDER = Path("ts054_extracted")


def load_ts054_csv():
    """Download and load the MSOA CSV from the TS054 ZIP file."""
    download_file(TS054_URL, TS054_FILE)
    TS054_FOLDER.mkdir(exist_ok=True)

    with ZipFile(TS054_FILE) as zip_file:
        msoa_files = [
            file_name
            for file_name in zip_file.namelist()
            if "msoa" in file_name.lower()
            and file_name.lower().endswith(".csv")
        ]

        if not msoa_files:
            raise ValueError(
                "Could not find an MSOA CSV in the TS054 ZIP file."
            )

        zip_file.extractall(TS054_FOLDER)

    csv_path = TS054_FOLDER / msoa_files[0]

    return pd.read_csv(
        csv_path,
        low_memory=False,
    )


def load_tenure_dataset():
    """Create council-renting and owner-occupation measures."""
    tenure_raw = load_ts054_csv()

    total_column = (
        "Tenure of household: Total: All households"
    )

    council_column = (
        "Tenure of household: Social rented: "
        "Rents from council or Local Authority"
    )

    owned_column = (
        "Tenure of household: Owned"
    )

    tenure = tenure_raw[
        [
            "geography code",
            total_column,
            council_column,
            owned_column,
        ]
    ].copy()

    tenure = tenure.rename(
        columns={
            "geography code": MAP_KEY,
            total_column: "total_households",
            council_column: "council_rented_households",
            owned_column: "owner_occupied_households",
        }
    )

    tenure[MAP_KEY] = clean_code(tenure[MAP_KEY])

    for column in [
        "total_households",
        "council_rented_households",
        "owner_occupied_households",
    ]:
        tenure[column] = to_number(tenure[column])

    denominator = tenure["total_households"]

    tenure["council_rented_percent"] = (
        100
        * tenure["council_rented_households"]
        / denominator
    ).where(denominator > 0)

    tenure["owner_occupied_percent"] = (
        100
        * tenure["owner_occupied_households"]
        / denominator
    ).where(denominator > 0)

    tenure = tenure[
        [
            MAP_KEY,
            "total_households",
            "council_rented_households",
            "council_rented_percent",
            "owner_occupied_households",
            "owner_occupied_percent",
        ]
    ]

    assert_unique(tenure, "Census tenure data")

    return tenure


# ============================================================
# 6. DATASET: MEDIAN HOUSE PRICE
# ============================================================

HOUSE_PRICE_URL = (
    "https://www.ons.gov.uk/file?uri="
    "/peoplepopulationandcommunity/housing/datasets/"
    "medianhousepricesbymiddlelayersuperoutputarea/"
    "yearendingseptember2025/"
    "medianpricepaidmsoa.xlsx"
)

HOUSE_PRICE_FILE = Path(
    "median_price_paid_msoa_september_2025.xlsx"
)


def load_house_price_dataset():
    """Load the ONS MSOA median price paid data."""
    download_file(HOUSE_PRICE_URL, HOUSE_PRICE_FILE)

    prices = pd.read_excel(
        HOUSE_PRICE_FILE,
        sheet_name="1a",
        header=2,
    )

    prices = prices[
        [
            "MSOA code",
            "Year ending Sep 2025",
        ]
    ].copy()

    prices = prices.rename(
        columns={
            "MSOA code": MAP_KEY,
            "Year ending Sep 2025": "median_house_price",
        }
    )

    prices[MAP_KEY] = clean_code(prices[MAP_KEY])
    prices["median_house_price"] = to_number(
        prices["median_house_price"]
    )

    assert_unique(prices, "house-price data")

    return prices


# ============================================================
# 7. DATASET: ONS POLICE FORCE AREA CRIME
# ============================================================

# This is a small ONS workbook, so it replaces the large Police.uk
# monthly archives. The crime layer is Police Force Area level, not MSOA.

POLICE_FORCE_CRIME_URL = (
    "https://www.ons.gov.uk/file?uri="
    "/peoplepopulationandcommunity/crimeandjustice/"
    "datasets/policeforceareadatatables/"
    "yearendingseptember2025/"
    "policeforceareatablesyesep25.xlsx"
)

POLICE_FORCE_CRIME_FILE = Path(
    "police_force_crime_year_ending_sep_2025.xlsx"
)

PFA_BOUNDARY_URL = (
    "https://open-geography-portalx-ons.hub.arcgis.com/"
    "api/download/v1/items/"
    "4823c6a18775400482137f05aa64f49b/"
    "geojson?layers=0"
)

PFA_BOUNDARY_FILE = Path(
    "police_force_area_boundaries.geojson"
)


def clean_name(series):
    """Create a forgiving join key for names from different sources."""
    return (
        series.astype("string")
        .str.upper()
        .str.replace("POLICE", "", regex=False)
        .str.replace("[^A-Z0-9]", "", regex=True)
    )


def load_pfa_boundaries():
    """Load official Police Force Area boundary polygons."""
    download_file(PFA_BOUNDARY_URL, PFA_BOUNDARY_FILE)
    boundaries = gpd.read_file(PFA_BOUNDARY_FILE)

    code_candidates = [
        column for column in boundaries.columns
        if "PFA" in column.upper()
        and "CD" in column.upper()
    ]
    name_candidates = [
        column for column in boundaries.columns
        if "PFA" in column.upper()
        and "NM" in column.upper()
    ]

    if not code_candidates or not name_candidates:
        raise ValueError(
            "Could not find PFA code and name columns in the "
            "boundary file."
        )

    boundaries = boundaries.rename(
        columns={
            code_candidates[0]: "PFA_CODE",
            name_candidates[0]: "PFA_NAME",
        }
    )

    boundaries["PFA_CODE"] = clean_code(
        boundaries["PFA_CODE"]
    )

    return boundaries.to_crs("EPSG:4326")


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        return [
            " | ".join(
                str(part)
                for part in column
                if str(part).lower() != "nan"
                and not str(part).startswith("Unnamed")
            ).strip()
            for column in df.columns
        ]

    return [str(column) for column in df.columns]


def load_pfa_crime_data():
    """Load overall recorded-crime rates from ONS Table P3."""
    download_file(
        POLICE_FORCE_CRIME_URL,
        POLICE_FORCE_CRIME_FILE,
    )

    # Read without headers first to locate the real header row.
    preview = pd.read_excel(
        POLICE_FORCE_CRIME_FILE,
        sheet_name="Table P3",
        header=None,
    )

    first_column = preview.iloc[:, 0].astype(str).str.strip()
    header_matches = preview.index[
        first_column.eq("Area Code")
    ].tolist()

    if not header_matches:
        raise ValueError(
            "Could not find the 'Area Code' header row in Table P3."
        )

    header_row = header_matches[0]

    crime = pd.read_excel(
        POLICE_FORCE_CRIME_FILE,
        sheet_name="Table P3",
        header=header_row,
    )

    # Remove line breaks from the ONS column labels.
    crime.columns = [
        " ".join(str(column).split())
        for column in crime.columns
    ]

    rate_candidates = [
        column for column in crime.columns
        if "Total recorded crime" in column
        and "excluding fraud" in column
    ]

    if not rate_candidates:
        raise ValueError(
            "Could not find the total recorded-crime rate column "
            "in Table P3."
        )

    rate_column = rate_candidates[0]

    crime = crime[
        [
            "Area Code",
            "Area Name",
            rate_column,
        ]
    ].copy()

    crime = crime.rename(
        columns={
            "Area Code": "PFA_CODE",
            "Area Name": "PFA_NAME_CRIME",
            rate_column: "recorded_crime_rate_per_1000",
        }
    )

    crime["PFA_CODE"] = clean_code(
        crime["PFA_CODE"]
    )

    crime["recorded_crime_rate_per_1000"] = to_number(
        crime["recorded_crime_rate_per_1000"]
    )

    # Keep Police Force Areas only, excluding national and regional totals.
    crime = crime[
        crime["PFA_CODE"].str.startswith("E23")
    ].copy()

    crime = crime.dropna(
        subset=[
            "PFA_CODE",
            "recorded_crime_rate_per_1000",
        ]
    )

    return crime[
        [
            "PFA_CODE",
            "PFA_NAME_CRIME",
            "recorded_crime_rate_per_1000",
        ]
    ].drop_duplicates("PFA_CODE")


# ============================================================
# 8. BUILD MAP DATASETS
# ============================================================


def build_map_data():
    """Load the MSOA data and the separate Police Force Area data."""
    msoa_data = load_boundary_dataset()

    msoa_datasets = [
        load_income_dataset(),
        load_tenure_dataset(),
        load_house_price_dataset(),
    ]

    for dataset in msoa_datasets:
        msoa_data = msoa_data.merge(
            dataset,
            on=MAP_KEY,
            how="left",
            validate="one_to_one",
        )

    pfa_boundaries = load_pfa_boundaries()
    pfa_crime = load_pfa_crime_data()

    pfa_data = pfa_boundaries.merge(
        pfa_crime,
        on="PFA_CODE",
        how="left",
        validate="one_to_one",
    )

    return {
        "msoa": msoa_data.to_crs("EPSG:4326"),
        "pfa": pfa_data.to_crs("EPSG:4326"),
    }


# ============================================================
# 8. GENERIC MAP HELPERS
# ============================================================


def format_value(value, format_type):
    if format_type == "currency":
        return f"£{value:,.0f}"

    if format_type == "percent":
        return f"{value:.0f}%"

    return f"{value:,.1f}"


def make_style_function(column, colour_scale):
    """Create a polygon style function for any indicator."""
    def style_area(feature):
        value = feature["properties"].get(column)

        if value is None or pd.isna(value):
            return {
                "fillColor": "#D3D3D3",
                "color": "white",
                "weight": 0.2,
                "fillOpacity": 0.2,
            }

        return {
            "fillColor": colour_scale(float(value)),
            "color": "white",
            "weight": 0.2,
            "fillOpacity": 0.6,
        }

    return style_area


def create_legend_block(
    legend_id,
    title,
    colour_scale,
    minimum,
    maximum,
    format_type,
):
    values = [
        minimum + (maximum - minimum) * i / 5
        for i in range(6)
    ]

    colour_stops = ", ".join(
        f"{colour_scale(value)} {i * 20}%"
        for i, value in enumerate(values)
    )

    midpoint = (minimum + maximum) / 2

    return f"""
    <div id="{html.escape(legend_id)}"
         style="display: none; margin-bottom: 8px;">
        <div style="font-weight: bold; margin-bottom: 5px;">
            {html.escape(title)}
        </div>
        <div style="
            height: 14px;
            width: 240px;
            background: linear-gradient(
                to right,
                {colour_stops}
            );
            border: 1px solid #777;
        "></div>
        <div style="
            display: flex;
            justify-content: space-between;
            margin-top: 4px;
        ">
            <span>{format_value(minimum, format_type)}</span>
            <span>{format_value(midpoint, format_type)}</span>
            <span>{format_value(maximum, format_type)}</span>
        </div>
    </div>
    """


def add_basemap(income_map):
    """Add the light basemap and labels."""
    folium.TileLayer(
        tiles=(
            "https://{s}.basemaps.cartocdn.com/"
            "light_nolabels/{z}/{x}/{y}{r}.png"
        ),
        attr=(
            "&copy; OpenStreetMap contributors "
            "&copy; CARTO"
        ),
        name="Light basemap",
        overlay=False,
        control=False,
    ).add_to(income_map)


def add_city_labels(income_map):
    """Add labels above the coloured polygons."""
    folium.TileLayer(
        tiles=(
            "https://{s}.basemaps.cartocdn.com/"
            "light_only_labels/{z}/{x}/{y}{r}.png"
        ),
        attr=(
            "&copy; OpenStreetMap contributors "
            "&copy; CARTO"
        ),
        name="City labels",
        overlay=True,
        control=False,
    ).add_to(income_map)


def add_focus_css(income_map):
    """Remove the orange focus outline after clicking."""
    income_map.get_root().header.add_child(
        folium.Element(
            """
            <style>
                .leaflet-container:focus,
                .leaflet-interactive:focus,
                .leaflet-overlay-pane path:focus {
                    outline: none !important;
                }
            </style>
            """
        )
    )


# ============================================================
# 9. BUILD THE MAP FROM THE CONFIGURATION
# ============================================================


def build_map(data_sources):
    income_map = folium.Map(
        location=[52.5, -1.5],
        zoom_start=6,
        tiles=None,
    )

    add_basemap(income_map)

    # Use the MSOA extent to set the initial map view.
    min_lon, min_lat, max_lon, max_lat = (
        data_sources["msoa"].total_bounds
    )

    income_map.fit_bounds(
        [
            [min_lat, min_lon],
            [max_lat, max_lon],
        ]
    )

    layer_refs = []
    legend_blocks = []

    for indicator_index, indicator_key in enumerate(
        ACTIVE_INDICATORS
    ):
        config = INDICATORS[indicator_key]
        column = config["column"]
        source_name = config.get("geo_source", "msoa")
        layer_data = data_sources[source_name]
        geojson_data = layer_data.to_json()

        if column not in layer_data.columns:
            raise ValueError(
                f"Column '{column}' is missing from {source_name} data."
            )

        colour_scale = getattr(
            linear,
            config["palette"],
        ).scale(
            config["minimum"],
            config["maximum"],
        )

        layer = folium.FeatureGroup(
            name=config["label"],
            show=(indicator_index == 0),
        )

        tooltip = folium.GeoJsonTooltip(
            fields=config.get(
                "tooltip_fields",
                ["Local authority name", column],
            ),
            aliases=config.get(
                "tooltip_aliases",
                ["Area:", config["tooltip_alias"]],
            ),
            localize=True,
        )

        folium.GeoJson(
            geojson_data,
            name=config["label"],
            style_function=make_style_function(
                column,
                colour_scale,
            ),
            tooltip=tooltip,
        ).add_to(layer)

        layer.add_to(income_map)

        legend_id = f"{indicator_key}-legend"

        legend_blocks.append(
            create_legend_block(
                legend_id=legend_id,
                title=config["legend_title"],
                colour_scale=colour_scale,
                minimum=config["minimum"],
                maximum=config["maximum"],
                format_type=config["format"],
            )
        )

        layer_refs.append(
            {
                "layer": layer,
                "legend_id": legend_id,
            }
        )

    add_city_labels(income_map)

    folium.LayerControl(
        collapsed=False
    ).add_to(income_map)

    legend_container = f"""
    <div id="dynamic-legends" style="
        position: fixed;
        bottom: 30px;
        left: 30px;
        z-index: 9999;
        background-color: white;
        border: 1px solid #999;
        border-radius: 4px;
        padding: 10px;
        width: 260px;
    ">
        {''.join(legend_blocks)}
    </div>
    """

    income_map.get_root().html.add_child(
        folium.Element(legend_container)
    )

    pair_javascript = ",\n".join(
        "{{layer: {layer}, legend: document.getElementById('{legend}')}}".format(
            layer=item["layer"].get_name(),
            legend=item["legend_id"],
        )
        for item in layer_refs
    )

    legend_script = f"""
    <script>
    document.addEventListener("DOMContentLoaded", function() {{
        const map = {income_map.get_name()};

        const layerLegendPairs = [
            {pair_javascript}
        ];

        function updateLegends() {{
            layerLegendPairs.forEach(function(pair) {{
                pair.legend.style.display = map.hasLayer(pair.layer)
                    ? "block"
                    : "none";
            }});
        }}

        updateLegends();
        map.on("overlayadd", updateLegends);
        map.on("overlayremove", updateLegends);
    }});
    </script>
    """

    income_map.get_root().html.add_child(
        folium.Element(legend_script)
    )

    add_focus_css(income_map)

    return income_map


# ============================================================
# 10. RUN THE PROJECT
# ============================================================

data_sources = build_map_data()
income_map = build_map(data_sources)
income_map.save(OUTPUT_FILE)